<div align="center">

# 🚕 Yandex Go Analytics

## 2 · Batch Ingestion

*Read Parquet and Avro files straight from the Volume and persist them in Bronze — as they are.*

![layer](https://img.shields.io/badge/layer-bronze-CD7F32)
![ingestion](https://img.shields.io/badge/ingestion-Auto%20Loader-FF3621)
![formats](https://img.shields.io/badge/formats-Parquet%20%2B%20Avro-00ADD4)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [1 · Setup](./1.%20Setup.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb) |

**Progress** &nbsp; ▰▰▱▱▱▱▱▱  **2 / 8**

---

## 🎯 Task requirements

- [x] Read the **`.parquet`** and **`.avro`** files directly from the Volume
- [x] Write them **"as-is"** into the `bronze` schema as **Delta** tables: `bronze.taxi` and `bronze.users`

## 🧩 How it is solved

Both tables use the same service, `BaseIngestionService`:

| Step | What happens |
|:--|:--|
| 1 · **Extract** | Auto Loader (`cloudFiles`) reads new files from the Volume; the schema is inferred and stored next to the checkpoint |
| 2 · **Audit columns** | `_ingested_at` (timestamp) and `_source_file` (path) are added — the business columns stay untouched |
| 3 · **Load** | Append to a Delta table with `trigger(availableNow=True)`: process everything available, then stop |

| Entity | Source | Target |
|:--|:--|:--|
| 🚕 Trips | `…/landing/parquet` | `bronze.taxi` |
| 👤 Users | `…/landing/avro` | `bronze.users` |

> 💡 **Incremental by design** — the checkpoint remembers which files were processed, so a re-run ingests only
> *new* files and never duplicates data. Auto Loader also adds a `_rescued_data` column that captures values
> that do not match the inferred schema.

## 1️⃣ Connect to Spark

In [1]:
from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-batch-ingestion")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 12:27:03.065 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-batch-ingestion | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_dev | env = env


## 2️⃣ Peek at the raw sources

Spark can read the files directly from the Volume path — no copy, no staging.

### 🚕 Parquet — trips

In [2]:
SRC_PARQUET = f"/Volumes/{CATALOG}/raw_files/landing/parquet"
SRC_AVRO = f"/Volumes/{CATALOG}/raw_files/landing/avro"

source_trips = spark.read.parquet(SRC_PARQUET)
print(f"Parquet source: {source_trips.count():,} rows | {len(source_trips.columns)} columns")
display(source_trips)

Parquet source: 20,000 rows | 22 columns


DataFrame[id: string, user_id: string, driver_id: string, vendor_id: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: bigint, trip_distance: double, rate_code_id: bigint, store_and_fwd_flag: string, pu_location_id: int, do_location_id: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: double]

### 👤 Avro — users

In [3]:
source_users = spark.read.format("avro").load(SRC_AVRO)
print(f"Avro source: {source_users.count():,} rows | {len(source_users.columns)} columns")
display(source_users)

Avro source: 4,000 rows | 5 columns


DataFrame[id: string, full_name: string, email: string, registration_date: date, is_plus_subscriber: boolean]

## 3️⃣ Ingest into Bronze

Two services, one pattern. Each call blocks until the available files are fully processed.

```text
 landing/avro      ──▶ UserBatchIngestionService      ──▶ bronze.users
 landing/parquet   ──▶ TaxiTripBatchIngestionService  ──▶ bronze.taxi
```

In [4]:
from src.core.etl.bronze import (
    TaxiTripBatchIngestionService,
    UserBatchIngestionService,
)

user_service = UserBatchIngestionService()
user_service.run(spark)

taxi_service = TaxiTripBatchIngestionService()
taxi_service.run(spark)

print("✅ Batch ingestion finished")

2026-10-09 12:27:10.727 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_dev.bronze.users
2026-10-09 12:27:10.728 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [avro] from: /Volumes/yandex_go_dev/raw_files/landing/avro
2026-10-09 12:27:10.924 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_dev.bronze.users
2026-10-09 12:27:29.648 | INFO     | src.core.etl.bronze.base:load:111 - Successfully loaded in yandex_go_dev.bronze.users
2026-10-09 12:27:29.651 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_dev.bronze.taxi
2026-10-09 12:27:29.652 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [parquet] from: /Volumes/yandex_go_dev/raw_files/landing/parquet
2026-10-09 12:27:29.654 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_dev.bronze.taxi
2026-10-09 12:27:45.978 | INFO     | src.core.etl.bronze.base:load:111 - Su

✅ Batch ingestion finished


---

## ✅ Checkpoint

- [x] Parquet and Avro files read directly from the Volume
- [x] `bronze.taxi` and `bronze.users` written as Delta tables
- [x] Row counts match the sources; re-running adds nothing

> **Next up ▶ [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb)** — stream drivers into Bronze, evolve the schema and travel through time.

<div align="center"><sub>Yandex Go Analytics · notebook 2 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>